In [1]:
!pip install lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 807.0 kB/s eta 0:00:0000:0100:01


In [23]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler

In [4]:
df = pd.read_csv('../data/Final Data/Chl-a/Chl-a-7-day.csv')

In [5]:
df['Chl-a'] = np.log1p(df['Chl-a'])

In [6]:
Q1 = df['Chl-a'].quantile(0.25)
Q3 = df['Chl-a'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df = df[(df['Chl-a'] >= lower_bound) & (df['Chl-a'] <= upper_bound)]

In [7]:
X = df.drop(columns=['Chl-a'])
y = df['Chl-a']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [11]:
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [12]:
monotonic_constraints = [1, 1, 1, 1, 0, 0, 0, 0, 0, 0]  

In [13]:
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

In [25]:
params = {
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "monotone_constraints": monotonic_constraints,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": -1,
    "verbose": -1,
}

lgb_model = lgb.train(params, train_data, valid_sets=[test_data], num_boost_round=100)

In [26]:
y_pred = lgb_model.predict(X_test)

In [27]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"R2 Score: {r2}")

MAE: 0.0783712249192223
MSE: 0.015148991998850614
RMSE: 0.12308124145803297
R2 Score: 0.68223059385383


# hyperparameter tuning

In [28]:
param_dist = {
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "num_leaves": [20, 31, 40, 50],
    "max_depth": [-1, 5, 10, 15],
    "n_estimators": [50, 100, 200, 500],
    "boosting_type": ["gbdt", "dart"],
}

In [29]:
lgb_model2 = lgb.LGBMRegressor(objective="regression", monotone_constraints=monotonic_constraints)

In [30]:
tuner = RandomizedSearchCV(lgb_model2, param_distributions=param_dist, n_iter=20, scoring='r2', cv=5, verbose=1, random_state=42, n_jobs=-1)
tuner.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002091 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 252
[LightGBM] [Info] Number of data points in the train set: 2194, number of used features: 10
[LightGBM] [Info] Start training from score 1.118050
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000676 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 252
[LightGBM] [Info] Number of data points in the train set: 2194, number of used features: 10
[LightGBM] [Info] Start training from score 1.118050
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further s

RandomizedSearchCV(cv=5,
                   estimator=LGBMRegressor(monotone_constraints=[1, 1, 1, 1, 0,
                                                                 0, 0, 0, 0,
                                                                 0],
                                           objective='regression'),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'boosting_type': ['gbdt', 'dart'],
                                        'learning_rate': [0.01, 0.05, 0.1, 0.2],
                                        'max_depth': [-1, 5, 10, 15],
                                        'n_estimators': [50, 100, 200, 500],
                                        'num_leaves': [20, 31, 40, 50]},
                   random_state=42, scoring='r2', verbose=1)

In [31]:
best_lgb_model = tuner.best_estimator_

In [32]:
best_lgb_model

LGBMRegressor(boosting_type='dart', learning_rate=0.2, max_depth=10,
              monotone_constraints=[1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
              n_estimators=500, num_leaves=50, objective='regression')

In [33]:
y_pred = best_lgb_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Best Parameters: {tuner.best_params_}")
print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")
print(f"R2 Score: {r2}")

Best Parameters: {'num_leaves': 50, 'n_estimators': 500, 'max_depth': 10, 'learning_rate': 0.2, 'boosting_type': 'dart'}
MAE: 0.07791668559810302
MSE: 0.014909396544034222
RMSE: 0.12210403983502849
R2 Score: 0.6872564137498405
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001533 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 252
[LightGBM] [Info] Number of data points in the train 